# 멀티모달 RAG — *Generative Modeling by Estimating Gradients of the Data Distribution*

- 분석 PDF: `data/sample.pdf` (Song & Ermon, arXiv:1907.05600, 23쪽)
- 구조: **1단계 추출 → 2단계 요약 → 3단계 요약 검색·원본 생성**
- 마지막에 같은 질문으로 **텍스트 RAG(Before) vs 멀티모달 RAG(After)** 를 비교합니다.

> 셀을 위에서 아래로 순서대로 실행하고, **출력이 남은 상태로 저장**한 뒤 커밋하세요.

## 0. 환경 점검

In [ ]:
import os
import shutil
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv()

PDF_PATH = Path("data/sample.pdf")
FIGURES_DIR = Path("figures")

print("Python   :", sys.version.split()[0])
print("PDF      :", "OK" if PDF_PATH.is_file() else "없음", PDF_PATH)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — PATH 확인 후 VS Code 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — PATH 확인 후 VS Code 재시작")

# 키 전체를 출력하면 노트북을 공유할 때 유출됩니다. 앞 4자리만 확인합니다.
key = os.getenv("OPENAI_API_KEY")
print("OPENAI   :", f"{key[:4]}****" if key else "✗ .env에 OPENAI_API_KEY 없음")

---
## 1단계. 추출 — 텍스트 · 표 · 이미지

- `strategy="hi_res"`: 레이아웃 모델로 표·그림 영역을 인식합니다. `fast`는 텍스트만 나옵니다.
- `infer_table_structure=True`: 표를 HTML로 보존합니다 (Table 1의 행·열 구조 유지).
- `extract_image_block_types=["Image", "Table"]`: 그림을 `figures/`에 jpg로 저장합니다.
- `chunking_strategy="by_title"`, `max_characters=1500`, `combine_text_under_n_chars=200`: 수업 실습 2와 같은 설정입니다.

In [ ]:
import time
from src.extract import extract_pdf

start = time.perf_counter()
extracted = extract_pdf(PDF_PATH, FIGURES_DIR)
texts, tables, images = extracted["texts"], extracted["tables"], extracted["images"]

print(f"소요 시간: {time.perf_counter() - start:.1f}초")
print("청크 카테고리 분포:", extracted["categories"])
print(f"텍스트 {len(texts)}개 / 표 {len(tables)}개 / 이미지 {len(images)}개")
print("figures/ 파일:", sorted(p.name for p in FIGURES_DIR.glob("*.jpg")))

In [ ]:
# 추출 결과 샘플 확인
print("[텍스트 샘플]", f"p.{texts[0]['page']}")
print(texts[0]["content"][:300], "\n")

for t in tables:
    print(f"[표 p.{t['page']}]")
    print(t["content"][:600], "\n")

In [ ]:
from IPython.display import Image, display

for img in images[:3]:
    print(img["image_path"], f"(p.{img['page']})")
    display(Image(filename=img["image_path"], width=360))

---
## 2단계. 요약 — 검색용 색인 만들기

- 모델: `gpt-4o-mini` (텍스트·표·이미지 모두 하나의 모델로 처리, 비전 입력 지원)
- 프롬프트 언어: **한국어 요약 + 수치·모델명은 원문 표기 유지** (질문을 한국어로 하므로)

In [ ]:
from src.summarize import summarize_texts, summarize_images

text_summaries = summarize_texts(texts)
table_summaries = summarize_texts(tables)
print(f"텍스트 요약 {len(text_summaries)}개 / 표 요약 {len(table_summaries)}개\n")

print("[텍스트 요약 샘플]", f"p.{texts[0]['page']}")
print(text_summaries[0], "\n")
for t, s in zip(tables, table_summaries):
    print(f"[표 요약 p.{t['page']}]")
    print(s, "\n")

In [ ]:
image_summaries = summarize_images(images)
print(f"이미지 요약 {len(image_summaries)}개\n")

for img, s in list(zip(images, image_summaries))[:5]:
    print(f"[{Path(img['image_path']).name} p.{img['page']}]")
    print(s, "\n")

---
## 3단계. 요약으로 검색 → 원본으로 생성

- **검색**은 Chroma에 넣은 *요약* 임베딩으로 합니다. 표 HTML·이미지는 그대로 임베딩하면 의미가 잘 안 잡히기 때문입니다.
- **생성**은 docstore에서 꺼낸 *원본*(본문, 표 HTML, 이미지 base64)으로 합니다. 요약 과정에서 빠진 수치·세부를 LLM이 직접 보게 하기 위해서입니다.

In [ ]:
from src.retriever import build_multi_vector_retriever
from src.chain import build_rag_chain, sources

mm_retriever = build_multi_vector_retriever(
    [(texts, text_summaries), (tables, table_summaries), (images, image_summaries)],
    k=4,
)
mm_chain = build_rag_chain(mm_retriever)

# 검색만 먼저 확인: 요약으로 찾았지만 돌려받는 것은 원본입니다.
docs = mm_retriever.invoke("CIFAR-10에서 NCSN의 Inception score와 FID")
for d in docs:
    preview = "(이미지 base64)" if d.metadata["kind"] == "image" else d.page_content[:120].replace("\n", " ")
    print(f"[{d.metadata['kind']} p.{d.metadata['page']}] {preview}")

In [ ]:
QUESTIONS = [
    "CIFAR-10 unconditional 설정에서 NCSN의 Inception score와 FID는 각각 얼마인가요?",
    "Table 1에서 Inception score가 NCSN보다 높은 모델은 무엇이고, 어떤 설정인가요?",
    "Figure 2에서 빨간 사각형으로 표시된 영역은 무엇을 의미하나요?",
    "Figure 3에서 annealed Langevin dynamics와 일반 Langevin dynamics의 결과는 어떻게 다른가요?",
    "이미지 생성 실험에서 노이즈 수준 σ는 몇 개, 어떤 범위로 설정했나요?",
]

mm_results = [mm_chain.invoke(q) for q in QUESTIONS]
for i, r in enumerate(mm_results, start=1):
    print(f"Q{i}. {r['question']}")
    print(r["answer"])
    print("검색된 원본:", sources(r["docs"]))
    print("-" * 80)

---
## Before / After — 텍스트 RAG vs 멀티모달 RAG

- **Before (텍스트 RAG)**: 같은 텍스트 청크 원문만 임베딩하고, 표·이미지는 쓰지 않습니다.
- **After (멀티모달 RAG)**: 위에서 만든 요약 검색 + 원본(텍스트·표·이미지) 생성 체인입니다.

In [ ]:
from src.retriever import build_text_retriever

text_chain = build_rag_chain(build_text_retriever(texts, k=4))
text_results = [text_chain.invoke(q) for q in QUESTIONS]

for i, (before, after) in enumerate(zip(text_results, mm_results), start=1):
    print(f"Q{i}. {before['question']}")
    print("[Before 텍스트 RAG]", before["answer"])
    print("   근거:", sources(before["docs"]))
    print("[After 멀티모달 RAG]", after["answer"])
    print("   근거:", sources(after["docs"]))
    print("=" * 80)

In [ ]:
# README에 붙여 넣을 질의응답 표 (After 기준)
def one_line(s):
    return s.replace("\n", " ").replace("|", "\\|")

print("| # | 질문 | 답변 | 근거가 된 원본 종류 |")
print("| --- | --- | --- | --- |")
for i, r in enumerate(mm_results, start=1):
    kinds = sorted({d.metadata["kind"] for d in r["docs"]})
    print(f"| {i} | {one_line(r['question'])} | {one_line(r['answer'])} | {', '.join(kinds)} ({sources(r['docs'])}) |")